# Voice-clone article narration with F5-TTS — Colab edition

This notebook generates one MP3 per article on `malakavenu.com` using **your own voice**, on Colab's free T4 GPU.

**You'll need 3 things ready before running:**

1. `articles.json` — produced locally by `npm run bake:prepare`. Upload it in step 2.
2. A `voice-sample.wav` (or .mp3) of you reading aloud for **15–30 seconds**, clean room, single speaker. Phone Voice Memos in a quiet room is fine.
3. The exact transcript of that sample as plain text (paste into step 2).

**End-to-end time:** ~5 min setup + ~30–45 min synthesis for 30 articles. Total $0.

## 1 · Sanity check the GPU

Runtime → Change runtime type → **T4 GPU** → Save. Then run the cell below; it should print a Tesla T4 (or better).

In [ ]:
!nvidia-smi -L

## 2 · Install F5-TTS and supporting tools

First run takes ~3–4 min — Colab caches wheels for repeats.

In [ ]:
!pip install -q f5-tts pydub
!apt-get -qq install -y ffmpeg > /dev/null

## 3 · Upload your voice sample and the article text

When the file picker pops up, select **both** files at once:

* your reference audio (`voice-sample.wav` / `.mp3` / `.m4a`)
* `articles.json` from `scripts/bake-audio/articles.json` in your repo (run `npm run bake:prepare` locally first)

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os

Path('/content/voice-sample').mkdir(exist_ok=True)
Path('/content/audio').mkdir(exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    if name.lower().endswith(('.wav', '.mp3', '.m4a', '.ogg', '.flac')):
        shutil.move(name, f'/content/voice-sample/{name}')
        print(f'  ref audio -> /content/voice-sample/{name}')
    elif name == 'articles.json':
        shutil.move(name, '/content/articles.json')
        print('  articles.json -> /content/articles.json')
    else:
        print(f'  ignored: {name}')


## 4 · Paste the EXACT transcript of your reference audio

F5-TTS uses this transcript to align prosody. It must be a verbatim transcription (including "um", filler words, etc., if you said them). 1–2 sentences are usually enough.

In [ ]:
REF_TEXT = """
Hi, I'm Malaka. I'm a frontend architect and AI engineer based in Bangalore.
I write about agent skills, MCP servers, and the design systems underneath.
""".strip()

Path('/content/voice-sample/ref_text.txt').write_text(REF_TEXT, encoding='utf-8')
print(f'Saved transcript ({len(REF_TEXT)} chars).')


## 5 · Bake every article to MP3

This is the slow cell. ~30–60 sec per article on a T4. Output goes to `/content/audio/<slug>.mp3`.

In [ ]:
import json, tempfile, time
from pathlib import Path
from f5_tts.api import F5TTS
from pydub import AudioSegment

VOICE_DIR = Path('/content/voice-sample')
OUT_DIR = Path('/content/audio')
MODEL_NAME = 'F5-TTS_v1'
MP3_BITRATE = '32k'   # mono speech — keeps each MP3 ~2-3 MB
FORCE = False         # set True to re-bake all

ref_audio = next(p for p in VOICE_DIR.iterdir()
                 if p.suffix.lower() in {'.wav','.mp3','.m4a','.ogg','.flac'})
ref_text = (VOICE_DIR / 'ref_text.txt').read_text(encoding='utf-8').strip()
items = json.loads(Path('/content/articles.json').read_text(encoding='utf-8'))

print(f'Loading {MODEL_NAME} (downloads ~1.5 GB on first run) ...')
tts = F5TTS(model=MODEL_NAME)
print(f'Reference voice: {ref_audio.name}')
print(f'Articles to bake: {len(items)}')

t0 = time.time()
for i, item in enumerate(items, 1):
    slug = item['slug']
    out_path = OUT_DIR / f'{slug}.mp3'
    if out_path.exists() and not FORCE:
        print(f'[{i:>2}/{len(items)}] {slug:50}  skip (exists)')
        continue
    text = item['text']
    print(f'[{i:>2}/{len(items)}] {slug:50}  {len(text)} chars ...', end='', flush=True)
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as wav_tmp:
        try:
            tts.infer(
                ref_file=str(ref_audio),
                ref_text=ref_text,
                gen_text=text,
                file_wave=wav_tmp.name,
                seed=42,
            )
            audio = AudioSegment.from_wav(wav_tmp.name).set_channels(1)
            audio.export(out_path, format='mp3', bitrate=MP3_BITRATE)
            kb = out_path.stat().st_size // 1024
            print(f'  done -> {out_path.name} ({kb} KB)')
        finally:
            Path(wav_tmp.name).unlink(missing_ok=True)

print(f'\nTotal time: {(time.time()-t0)/60:.1f} min')


## 6 · Quick listening test

Plays the first 60 seconds of one article so you can confirm the voice quality before downloading 30 files.

In [ ]:
from IPython.display import Audio, display
first = sorted(OUT_DIR.glob('*.mp3'))[0]
print(f'Previewing {first.name}')
display(Audio(filename=str(first)))


## 7 · Download all MP3s as a single zip

Unzip into your repo's `public/audio/` folder, commit, push. Vercel deploys; the Listen button will now play your voice automatically (the hook tries `/audio/<slug>.mp3` first).

In [ ]:
import shutil
shutil.make_archive('/content/audio', 'zip', root_dir='/content/audio')
from google.colab import files
files.download('/content/audio.zip')
